In [1]:
import numpy as np, pandas as pd, lightgbm as lgb, gc, time

data_path = r"C:\Users\Hp\OneDrive\Desktop\amazon hackthon\student_resource\dataset\train"
gt = pd.read_csv(data_path + r"\train_ground_truth.tsv", sep="\t", dtype=str)
true = gt.dropna(subset=["matched_entity_ids"]).copy()
true["match_id"] = true["matched_entity_ids"].str.split(",")
true = true.explode("match_id")
n_true_of = true.groupby("source1_entity_id").size()     # number of real matches per S1
print("S1 with matches:", len(n_true_of))

S1 with matches: 2083574


In [2]:
FEATURES = ["n_keys", "p1", "rank1", "name_tset", "name_ratio", "addr_tset",
            "name_tsort", "name_partial", "name_jw", "name_join",
            "addr_ratio", "addr_tsort", "addr_partial", "addr_jw",
            "house_eq", "num_jacc", "num_conflict", "legal_cmp",
            "len_name_s1", "len_name_o", "len_addr_s1", "len_addr_o", "is_s3",
            "n_cand", "p1_gap", "name_tset_gap", "addr_tset_gap"]

rng = np.random.default_rng(7)
parts = {}
for c in ["US", "India"]:
    d = dict(np.load(f"features_train_{c}.npz", allow_pickle=True))
    n_s1 = len(d["ids_s1"])
    usable = ~d["is_train_s1"]                         # not used by the stage-1 model
    is_val = usable & (rng.random(n_s1) < 0.25)       # 25% of them for validation
    d["s1_split"] = np.where(is_val, "val", np.where(usable, "train", "skip"))
    d["X"] = np.column_stack([d[f].astype(np.float32) for f in FEATURES])
    parts[c] = d
    print(c, {k: int((d["s1_split"] == k).sum()) for k in ["train", "val", "skip"]})

X_tr, y_tr, X_va, y_va = [], [], [], []
for c, d in parts.items():
    split = d["s1_split"][d["i"]]                      # split of each pair = split of its S1
    tr, va = np.flatnonzero(split == "train"), np.flatnonzero(split == "val")
    if len(tr) > 6_000_000:                            # at most 6M training pairs per country
        tr = np.sort(rng.choice(tr, 6_000_000, replace=False))
    X_tr.append(d["X"][tr]); y_tr.append(d["is_true"][tr])
    X_va.append(d["X"][va]); y_va.append(d["is_true"][va])
X_tr, y_tr = np.vstack(X_tr), np.concatenate(y_tr)
X_va, y_va = np.vstack(X_va), np.concatenate(y_va)
print("train pairs:", len(y_tr), " val pairs:", len(y_va))

model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=127,
                           min_child_samples=100, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, verbose=-1)
model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="binary_logloss",
          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
del X_tr, y_tr, X_va, y_va; gc.collect()
model.booster_.save_model("match_model.txt")
print("model saved")

US {'train': 794159, 'val': 264765, 'skip': 264709}
India {'train': 529546, 'val': 177151, 'skip': 176491}
train pairs: 12000000  val pairs: 6190021


c:\Users\Hp\OneDrive\Desktop\amazon hackthon\amazon\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.0703498
[200]	valid_0's binary_logloss: 0.0605641
[300]	valid_0's binary_logloss: 0.0574421
[400]	valid_0's binary_logloss: 0.0562867
Early stopping, best iteration is:
[410]	valid_0's binary_logloss: 0.0561466
model saved


In [3]:
def val_predictions():
    rows = []
    for c, d in parts.items():
        va = np.flatnonzero(d["s1_split"][d["i"]] == "val")
        prob = model.predict_proba(d["X"][va])[:, 1]
        rows.append(pd.DataFrame({"s1": d["ids_s1"][d["i"][va]], "prob": prob, "is_true": d["is_true"][va]}))
    # ALL validation S1s, including those with no candidates and singletons
    val_s1 = np.concatenate([d["ids_s1"][d["s1_split"] == "val"] for d in parts.values()])
    return pd.concat(rows, ignore_index=True), val_s1

def f05_score(pred_df, val_s1, t):
    n_true = n_true_of.reindex(val_s1).fillna(0).to_numpy()
    idx = pd.Index(val_s1)
    chosen = pred_df[pred_df["prob"] >= t]                      # predicted matches
    pos = idx.get_indexer(chosen["s1"])
    pp = np.bincount(pos, minlength=len(val_s1))                 # predicted per S1
    tp = np.bincount(pos, weights=chosen["is_true"].to_numpy(), minlength=len(val_s1))  # correct per S1
    P = np.divide(tp, pp, out=np.zeros(len(tp)), where=pp > 0)
    R = np.divide(tp, n_true, out=np.zeros(len(tp)), where=n_true > 0)
    F = np.divide(1.25 * P * R, 0.25 * P + R, out=np.zeros(len(tp)), where=(0.25 * P + R) > 0)
    F[(n_true == 0) & (pp == 0)] = 1.0                           # singleton, predicted nothing = perfect
    return F.mean()

pred_df, val_s1 = val_predictions()
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    print(f"threshold {t}: F0.5 = {100 * f05_score(pred_df, val_s1, t):.2f}%")

threshold 0.3: F0.5 = 90.89%
threshold 0.4: F0.5 = 92.02%
threshold 0.5: F0.5 = 92.83%
threshold 0.6: F0.5 = 93.25%
threshold 0.7: F0.5 = 93.28%
threshold 0.8: F0.5 = 92.89%
threshold 0.9: F0.5 = 91.62%


In [4]:
def per_s1_scores(chosen, val_s1):
    n_true = n_true_of.reindex(val_s1).fillna(0).to_numpy()
    idx = pd.Index(val_s1)
    pos = idx.get_indexer(chosen["s1"])
    pp = np.bincount(pos, minlength=len(val_s1))
    tp = np.bincount(pos, weights=chosen["is_true"].to_numpy(), minlength=len(val_s1))
    P = np.divide(tp, pp, out=np.zeros(len(tp)), where=pp > 0)
    R = np.divide(tp, n_true, out=np.zeros(len(tp)), where=n_true > 0)
    F = np.divide(1.25 * P * R, 0.25 * P + R, out=np.zeros(len(tp)), where=(0.25 * P + R) > 0)
    F[(n_true == 0) & (pp == 0)] = 1.0
    return F, n_true, pp, tp

pred_df["max_p"] = pred_df.groupby("s1")["prob"].transform("max")
n_cand_true = pred_df.groupby("s1")["is_true"].sum().reindex(val_s1).fillna(0).to_numpy()

# ---- where is the score lost? (threshold 0.7) ----
F, n_true, pp, tp = per_s1_scores(pred_df[pred_df["prob"] >= 0.7], val_s1)
N = len(val_s1)
groups = {
    "singleton, but we predicted a match":        (n_true == 0) & (pp > 0),
    "has matches, none in candidates (blocking)": (n_true > 0) & (n_cand_true == 0),
    "has matches in candidates, predicted none":  (n_true > 0) & (n_cand_true > 0) & (pp == 0),
    "predicted some, but with wrong ones (FP)":   (n_true > 0) & (pp > 0) & (tp < pp),
    "predicted only right ones, missed some":     (n_true > 0) & (pp > 0) & (tp == pp) & (tp < n_true),
}
print(f"score {100 * F.mean():.2f}%  → lost {100 * (1 - F.mean()):.2f}% in total")
for name, m in groups.items():
    print(f"  {name:45s}: {m.sum():>7,} S1  lose {100 * (1 - F[m]).sum() / N:5.2f}%")

# ---- try smarter decision rules ----
print("\nrule: keep if prob >= t AND prob >= r * best prob of that S1")
for t in [0.5, 0.6, 0.7]:
    for r in [0.0, 0.5, 0.8]:
        ch = pred_df[(pred_df["prob"] >= t) & (pred_df["prob"] >= r * pred_df["max_p"])]
        print(f"  t={t} r={r}: {100 * per_s1_scores(ch, val_s1)[0].mean():.2f}%")

print("\nrule: best candidate needs only t_best, others need t")
is_best = pred_df["prob"] == pred_df["max_p"]
for t_best in [0.4, 0.5, 0.6]:
    for t in [0.6, 0.7, 0.8]:
        ch = pred_df[(is_best & (pred_df["prob"] >= t_best)) | (pred_df["prob"] >= t)]
        print(f"  t_best={t_best} t={t}: {100 * per_s1_scores(ch, val_s1)[0].mean():.2f}%")

score 93.28%  → lost 6.72% in total
  singleton, but we predicted a match          :   1,643 S1  lose  0.37%
  has matches, none in candidates (blocking)   :   3,005 S1  lose  0.68%
  has matches in candidates, predicted none    :   4,607 S1  lose  1.04%
  predicted some, but with wrong ones (FP)     :  21,514 S1  lose  1.37%
  predicted only right ones, missed some       : 133,065 S1  lose  3.29%

rule: keep if prob >= t AND prob >= r * best prob of that S1
  t=0.5 r=0.0: 92.83%
  t=0.5 r=0.5: 92.83%
  t=0.5 r=0.8: 93.09%
  t=0.6 r=0.0: 93.25%
  t=0.6 r=0.5: 93.25%
  t=0.6 r=0.8: 93.17%
  t=0.7 r=0.0: 93.28%
  t=0.7 r=0.5: 93.28%
  t=0.7 r=0.8: 93.11%

rule: best candidate needs only t_best, others need t
  t_best=0.4 t=0.6: 92.98%
  t_best=0.4 t=0.7: 93.04%
  t_best=0.4 t=0.8: 92.83%
  t_best=0.5 t=0.6: 93.16%
  t_best=0.5 t=0.7: 93.22%
  t_best=0.5 t=0.8: 93.00%
  t_best=0.6 t=0.6: 93.25%
  t_best=0.6 t=0.7: 93.31%
  t_best=0.6 t=0.8: 93.10%


In [5]:
m = groups["predicted only right ones, missed some"]
missed_total = (n_true - tp)[m].sum()
missed_in_cand = (n_cand_true - tp)[m].sum()
print(f"missed matches in these S1s: {int(missed_total):,}")
print(f"  in candidates, but model said < 0.7: {int(missed_in_cand):,} ({100 * missed_in_cand / missed_total:.1f}%)")
print(f"  not in candidates at all:            {int(missed_total - missed_in_cand):,}")

missed matches in these S1s: 184,149
  in candidates, but model said < 0.7: 114,833 (62.4%)
  not in candidates at all:            69,316


In [6]:
from rapidfuzz import process, fuzz

needed = set()
for d in parts.values():
    needed.update(d["ids_other"])
cols = ["entity_id", "name_core", "addr_clean"]
text = pd.concat([
    pd.read_csv(f"{s}_clean.tsv", sep="\t", dtype=str, keep_default_na=False, usecols=cols)
      .loc[lambda df: df["entity_id"].isin(needed)]
    for s in ["s2", "s3"]
]).set_index("entity_id")
del needed; gc.collect()
print("text loaded:", text.shape)

text loaded: (10320219, 2)


In [7]:
N_FOLDS = 3
best_iter = model.best_iteration_ or 400

for c, d in parts.items():
    d["prob"] = np.full(len(d["i"]), np.nan, np.float32)
    va = np.flatnonzero(d["s1_split"][d["i"]] == "val")
    d["prob"][va] = model.predict_proba(d["X"][va])[:, 1]          # validation: current model
    d["fold"] = rng.integers(0, N_FOLDS, len(d["ids_s1"]))          # each S1 gets a fold number

for f in range(N_FOLDS):
    X_tr, y_tr = [], []
    for c, d in parts.items():
        is_tr = d["s1_split"][d["i"]] == "train"
        rows = np.flatnonzero(is_tr & (d["fold"][d["i"]] != f))     # train on the OTHER folds
        if len(rows) > 4_000_000:
            rows = np.sort(rng.choice(rows, 4_000_000, replace=False))
        X_tr.append(d["X"][rows]); y_tr.append(d["is_true"][rows])
    m = lgb.LGBMClassifier(n_estimators=best_iter, learning_rate=0.05, num_leaves=127,
                           min_child_samples=100, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, verbose=-1)
    m.fit(np.vstack(X_tr), np.concatenate(y_tr))
    del X_tr, y_tr; gc.collect()
    for c, d in parts.items():                                       # predict THIS fold
        rows = np.flatnonzero((d["s1_split"][d["i"]] == "train") & (d["fold"][d["i"]] == f))
        d["prob"][rows] = m.predict_proba(d["X"][rows])[:, 1]
    print(f"fold {f + 1}/{N_FOLDS} done")

fold 1/3 done
fold 2/3 done
fold 3/3 done


In [8]:
def sibling_features(d, anchor_t=0.8, chunk_s1=200_000):
    i, j, prob = d["i"], d["j"], d["prob"]
    n = len(i)
    usable = ~np.isnan(prob)
    nameo = text.loc[d["ids_other"], "name_core"].to_numpy()
    addro = text.loc[d["ids_other"], "addr_clean"].to_numpy()

    anchor = usable & (prob >= anchor_t)                                 # confident matches
    n_anchor = np.bincount(i[anchor], minlength=len(d["ids_s1"]))
    d["n_conf"] = (n_anchor[i] - anchor).astype(np.int8)                 # not counting itself
    d["sib_name"] = np.zeros(n, np.uint8)
    d["sib_addr"] = np.zeros(n, np.uint8)

    A = pd.DataFrame({"i": i[anchor], "aj": j[anchor]})
    rows_all = np.flatnonzero(usable)
    for lo in range(0, len(d["ids_s1"]), chunk_s1):                      # a block of S1s at a time
        hi = lo + chunk_s1
        r = rows_all[(i[rows_all] >= lo) & (i[rows_all] < hi)]
        P = pd.DataFrame({"row": r, "i": i[r], "j": j[r]})
        M = P.merge(A[(A["i"] >= lo) & (A["i"] < hi)], on="i")            # each candidate × each confident match
        M = M[M["j"] != M["aj"]]
        if len(M) == 0:
            continue
        s_n = process.cpdist(nameo[M["j"].to_numpy()], nameo[M["aj"].to_numpy()],
                             scorer=fuzz.token_set_ratio, workers=-1, dtype=np.uint8)
        s_a = process.cpdist(addro[M["j"].to_numpy()], addro[M["aj"].to_numpy()],
                             scorer=fuzz.token_set_ratio, workers=-1, dtype=np.uint8)
        np.maximum.at(d["sib_name"], M["row"].to_numpy(), s_n)          # keep the best similarity
        np.maximum.at(d["sib_addr"], M["row"].to_numpy(), s_a)
    print("done:", round(float((d["n_conf"] > 0).mean()), 3), "of pairs have a confident sibling")

for c, d in parts.items():
    sibling_features(d)

done: 0.742 of pairs have a confident sibling
done: 0.727 of pairs have a confident sibling


In [9]:
for c, d in parts.items():
    d["X2"] = np.column_stack([d["X"], d["sib_name"].astype(np.float32),
                               d["sib_addr"].astype(np.float32), d["n_conf"].astype(np.float32)])
    del d["X"]; gc.collect()                          # free memory

X_tr, y_tr, X_va, y_va = [], [], [], []
for c, d in parts.items():
    split = d["s1_split"][d["i"]]
    tr, va = np.flatnonzero(split == "train"), np.flatnonzero(split == "val")
    if len(tr) > 6_000_000:
        tr = np.sort(rng.choice(tr, 6_000_000, replace=False))
    X_tr.append(d["X2"][tr]); y_tr.append(d["is_true"][tr])
    X_va.append(d["X2"][va]); y_va.append(d["is_true"][va])
X_tr, y_tr = np.vstack(X_tr), np.concatenate(y_tr)
X_va, y_va = np.vstack(X_va), np.concatenate(y_va)

model2 = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=127,
                            min_child_samples=100, subsample=0.8, subsample_freq=1,
                            colsample_bytree=0.8, verbose=-1)
model2.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="binary_logloss",
           callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
del X_tr, y_tr, X_va, y_va; gc.collect()
model2.booster_.save_model("match_model2.txt")

rows = []
for c, d in parts.items():
    va = np.flatnonzero(d["s1_split"][d["i"]] == "val")
    rows.append(pd.DataFrame({"s1": d["ids_s1"][d["i"][va]],
                              "prob": model2.predict_proba(d["X2"][va])[:, 1],
                              "is_true": d["is_true"][va]}))
pred_df2 = pd.concat(rows, ignore_index=True)
for t in [0.5, 0.6, 0.7, 0.8]:
    print(f"threshold {t}: F0.5 = {100 * f05_score(pred_df2, val_s1, t):.2f}%   (before: 93.28% best)")

c:\Users\Hp\OneDrive\Desktop\amazon hackthon\amazon\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.0658157
[200]	valid_0's binary_logloss: 0.0557658
[300]	valid_0's binary_logloss: 0.052467
[400]	valid_0's binary_logloss: 0.0508786
[500]	valid_0's binary_logloss: 0.0498839
[600]	valid_0's binary_logloss: 0.0492339
[700]	valid_0's binary_logloss: 0.0485219
[800]	valid_0's binary_logloss: 0.0480911
Early stopping, best iteration is:
[798]	valid_0's binary_logloss: 0.0480903
threshold 0.5: F0.5 = 93.48%   (before: 93.28% best)
threshold 0.6: F0.5 = 93.74%   (before: 93.28% best)
threshold 0.7: F0.5 = 93.74%   (before: 93.28% best)
threshold 0.8: F0.5 = 93.36%   (before: 93.28% best)


In [10]:
import pandas as pd, csv
test_path = r"C:\Users\Hp\OneDrive\Desktop\amazon hackthon\student_resource\dataset\test"
for f in ["test_source1.tsv", "test_source2.tsv", "test_source3.tsv"]:
    df = pd.read_csv(test_path + "\\" + f, sep="\t", dtype=str, quoting=csv.QUOTE_NONE)
    print(f, df.shape, df["country"].value_counts().to_dict())

test_source1.tsv (1732544, 4) {'India': 809986, 'US': 663106, 'France': 259452}
test_source2.tsv (4887273, 4) {'India': 2312565, 'US': 1871330, 'France': 703378}
test_source3.tsv (5082316, 4) {'India': 2405000, 'US': 1945701, 'France': 731615}


In [11]:
import os
for f in ["predictions_test_France.npz", "predictions_test_India.npz", "predictions_test_US.npz",
          "output/matching_results.tsv", "output/candidate_pairs.tsv"]:
    print(f, "✓" if os.path.exists(f) else "✗ missing")

predictions_test_France.npz ✓
predictions_test_India.npz ✓
predictions_test_US.npz ✓
output/matching_results.tsv ✗ missing
output/candidate_pairs.tsv ✗ missing


In [12]:
from pipeline import *

test_dir = r"C:\Users\Hp\OneDrive\Desktop\amazon hackthon\student_resource\dataset\test"
countries = ["France", "India", "US"]

step7_write(countries, test_dir)

KeyboardInterrupt: 